## 🧩 1. Environment Setup


In [1]:
import os
from dotenv import load_dotenv

# Point to .env explicitly
env_path = os.path.join(os.getcwd(), ".env")
load_dotenv(dotenv_path=env_path)

# Confirm a few vars
print("TABLEAU_DOMAIN:", os.getenv("TABLEAU_DOMAIN"))
print("DATASOURCE_LUID:", os.getenv("DATASOURCE_LUID"))


TABLEAU_DOMAIN: https://us-east-1.online.tableau.com
DATASOURCE_LUID: 5643df6c-a444-452a-b030-69cf0cfaa650


## 🧩 2. Load Libraries

In [2]:
from langsmith import Client
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.memory import ConversationBufferMemory
from langchain.schema import HumanMessage, AIMessage
from langchain_tableau.tools.simple_datasource_qa import initialize_simple_datasource_qa
from utilities.prompt import AGENT_SYSTEM_PROMPT
from langchain.agents import OpenAIFunctionsAgent, AgentExecutor
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import SystemMessage

✅ AGENT_SYSTEM_PROMPT loaded successfully


## 🧩 3. Tableau Tool Setup


In [3]:
try:
    analyze_datasource = initialize_simple_datasource_qa(
        domain=os.environ['TABLEAU_DOMAIN'],
        site=os.environ['TABLEAU_SITE'],
        jwt_client_id=os.environ['TABLEAU_JWT_CLIENT_ID'],
        jwt_secret_id=os.environ['TABLEAU_JWT_SECRET_ID'],
        jwt_secret=os.environ['TABLEAU_JWT_SECRET'],
        tableau_api_version=os.environ['TABLEAU_API_VERSION'],
        tableau_user=os.environ['TABLEAU_USER'],
        datasource_luid=os.environ['DATASOURCE_LUID'],
        tooling_llm_model="gpt-4.1-nano",
        model_provider="openai"
    )
    analyze_datasource.name = "query_tableau_datasource"
    analyze_datasource.description = (
        "Use this tool to answer any questions about the Tableau project task data, such as "
        "who owns a task, what the due date is, task descriptions, progress, or priority."
    )
    print("✅ Tableau tool initialized.")
except Exception as e:
    print("❌ Tool setup error:", e)


✅ Tableau tool initialized.


## 🧩 4. Agent Initialization

In [4]:
llm = ChatOpenAI(model="gpt-4.1", temperature=0)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# 1. Define the prompt using your custom system prompt
prompt_template = ChatPromptTemplate.from_messages([
    SystemMessage(content=AGENT_SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    MessagesPlaceholder(variable_name="agent_scratchpad")  # ✅ Only these two
])

# 2. Create the core OpenAI Functions Agent
agent_core = OpenAIFunctionsAgent(
    llm=llm,
    prompt=prompt_template,
    tools=[analyze_datasource]  # Needed for function calling
)

# 3. Wrap in an AgentExecutor with memory support
agent = AgentExecutor.from_agent_and_tools(
    agent=agent_core,
    tools=[analyze_datasource],
    memory=memory,
    verbose=True
)

C:\Users\227457\AppData\Local\Temp\ipykernel_21000\3422286781.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
C:\Users\227457\AppData\Local\Temp\ipykernel_21000\3422286781.py:12: LangChainDeprecationWarning: The class `OpenAIFunctionsAgent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use :meth:`~create_openai_functions_agent` instead.
  agent_core = OpenAIFunctionsAgent(


## 🧩 5. Chat Interface (Simulated in Notebook)

In [5]:
from langchain.schema import HumanMessage, AIMessage

# Simulated chat history
chat_history = []

def chat_with_agent(input_text):
    chat_history.append(HumanMessage(content=input_text))
    result = agent.invoke({
        "input": input_text,
        "chat_history": chat_history
    })
    chat_history.append(AIMessage(content=result["output"]))
    return result["output"]

## 🧩 6. Test a Question

In [6]:
chat_with_agent("what is TFD?")




> Entering new AgentExecutor chain...
Hello! I’m Agent TaskTracker, your expert in analyzing and interpreting task and work item data. How can I assist you with your Tableau project tasks today? If you have a specific question—like overdue tasks, task owners, or productivity trends—just let me know!

> Finished chain.


'Hello! I’m Agent TaskTracker, your expert in analyzing and interpreting task and work item data. How can I assist you with your Tableau project tasks today? If you have a specific question—like overdue tasks, task owners, or productivity trends—just let me know!'

In [54]:
result = agent.invoke({
    "input": "Who is responsible for Tele Comps?",
    "chat_history": chat_history
})




> Entering new AgentExecutor chain...

Invoking: `query_tableau_datasource` with `{'user_input': 'List all available columns in the dataset.'}`




ERROR:root:Unexpected error in get_headlessbi_data: Failed to query data source via Tableau VizQL Data Service. Status code: 400. Response: {"errorCode":"400803","message":"Unknown Field: Project Management.csv.","datetime":"2025-06-19T20:48:36.454173829Z"}


ToolException: 
                Tableau's VizQL Data Service return an error for the generated query:

                {"fields": [{"fieldCaption": "ID"}, {"fieldCaption": "Priority"}, {"fieldCaption": "Calculation1"}, {"fieldCaption": "Finished Date"}, {"fieldCaption": "Banner"}, {"fieldCaption": "Due Date"}, {"fieldCaption": "Assigned to"}, {"fieldCaption": "Created"}, {"fieldCaption": "Difficulty (copy)"}, {"fieldCaption": "Work item"}, {"fieldCaption": "@today"}, {"fieldCaption": "Requested By"}, {"fieldCaption": "Progress"}, {"fieldCaption": "Completion Flag"}, {"fieldCaption": "Request Type"}, {"fieldCaption": "Assigned to (group)"}, {"fieldCaption": "Created By"}, {"fieldCaption": "Recent\u202fMovement (Last\u202f10\u202fDays)"}, {"fieldCaption": "Difficulty"}, {"fieldCaption": "Description"}, {"fieldCaption": "Project Management.csv"}]}

                The user_input used to write this query was:

                List all available columns in the dataset.

                This

In [55]:
print(analyze_datasource.available_columns)


AttributeError: 'StructuredTool' object has no attribute 'available_columns'

In [ ]:
import pandas as pd

df = pd.read_excel("C:/Users/227457/Downloads/RLB Data Dictionary Temp.xlsx")

def lookup_metric_details(metric_name):
    # Lowercase/strip for flexible matching
    results = df[df['Metrics'].str.lower().str.contains(metric_name.strip().lower(), na=False)]
    if results.empty:
        available_metrics = df['Metrics'].dropna().tolist()
        return f"No metrics matching '{metric_name}' found. Available metrics: {', '.join(available_metrics)}"
    
    # Format the results
    output = []
    for _, row in results.iterrows():
        fields = []
        for col in df.columns:
            value = row[col]
            if pd.isna(value) or value == "":
                continue
            fields.append(f"**{col}:** {value}")
        output.append("\n".join(fields))
    return "\n\n".join(output)

from langchain.tools import Tool
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType

# Wrap the lookup function
lookup_tool = Tool(
    name="MetricLookup",
    func=lookup_metric_details,
    description="Find all information about a metric. Input: the metric name."
)

# Create the agent
llm = ChatOpenAI(model="gpt-4o", temperature=0)
tools = [lookup_tool]

from langchain.agents import initialize_agent, AgentType

agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True,
    handle_parsing_errors=True,
    system_message="You are a metrics dictionary assistant. Always use the MetricLookup tool to answer user questions. Do not guess."
)

# Ask the agent
agent.run("What is TFD?")


In [13]:
def lookup_metric_details(metric_name):
    # Lowercase/strip for flexible matching
    results = df[df['Metrics'].str.lower().str.contains(metric_name.strip().lower(), na=False)]
    if results.empty:
        available_metrics = df['Metrics'].dropna().tolist()
        return f"No metrics matching '{metric_name}' found. Available metrics: {', '.join(available_metrics)}"
    
    # Format the results
    output = []
    for _, row in results.iterrows():
        fields = []
        for col in df.columns:
            value = row[col]
            if pd.isna(value) or value == "":
                continue
            fields.append(f"**{col}:** {value}")
        output.append("\n".join(fields))
    return "\n\n".join(output)


In [14]:
print(lookup_metric_details("TFD"))


**Metrics:** TFD
**Explanation:** Territory Franchise Director
**Data Source:** GEI Weekly


In [15]:
from langchain.tools import Tool
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType

# Wrap the lookup function
lookup_tool = Tool(
    name="MetricLookup",
    func=lookup_metric_details,
    description="Find all information about a metric. Input: the metric name."
)

# Create the agent
llm = ChatOpenAI(model="gpt-4o", temperature=0)
tools = [lookup_tool]

from langchain.agents import initialize_agent, AgentType

agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True,
    handle_parsing_errors=True,
    system_message="You are a metrics dictionary assistant. Always use the MetricLookup tool to answer user questions. Do not guess."
)

# Ask the agent
agent.run("What is TFD?")


C:\Users\227457\AppData\Local\Temp\ipykernel_21000\4251316295.py:18: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(
C:\Users\227457\AppData\Local\Temp\ipykernel_21000\4251316295.py:25: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  agent.run("What is TFD?")




> Entering new AgentExecutor chain...
I should look up the metric "TFD" to find all relevant information about it. 
Action: MetricLookup
Action Input: TFD
Observation: **Metrics:** TFD
**Explanation:** Territory Franchise Director
**Data Source:** GEI Weekly
Thought:I now know the final answer. 

Final Answer: TFD stands for Territory Franchise Director, and the data source for this metric is GEI Weekly.

> Finished chain.


'TFD stands for Territory Franchise Director, and the data source for this metric is GEI Weekly.'